# Knobs

In [1]:
min_stitch = 2
min_off = 0.1
seg_step = 0.1 #90% overlap, seg set to 10s strides
clus_th = 0.45

# Imports PS: Dependencies have already been installed through the dependency manager

In [2]:
import torch
import torchaudio
from tqdm.auto import tqdm
from pyannote.audio import Pipeline, Model
import json
import pandas as pd
import os
import gc

# File Path Config

In [3]:
pipeline_path = "/kaggle/input/models/gomiie/pyannote-community-1/pytorch/default/1/pyannote_community_model/config.yaml"
seg_path = "/kaggle/input/models/gomiie/segmentation-3-0-finetuned-bangla/pytorch/default/1/pytorch_model.bin"
emb_path = "/kaggle/input/models/gomiie/wespeaker-voxceleb-resnet34-lm/pytorch/default/1/wespeaker/pytorch_model.bin"

wav_path = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio"
all_files = sorted([f for f in os.listdir(wav_path) if f.endswith(".wav")])
OUT_PATH = "submission.csv" 

# Pipeline + Model Setup

In [4]:
device = torch.device("cuda")

pipeline = Pipeline.from_pretrained(pipeline_path).to(device)
seg_model = Model.from_pretrained(seg_path).eval().to(device)
emb_model = Model.from_pretrained(emb_path).eval().to(device)

#Replacing their segmentation model with our bangla finetuned one
pipeline.segmentation = seg_model
pipeline._segmentation.model = seg_model

#Replacing their pyannote embedding model with the wespeaker resned34 one
pipeline.embedding = emb_model
pipeline._embedding.model = emb_model

pipeline.segmentation_step = seg_step
pipeline.segmentation.min_duration_off = min_off
pipeline.clustering.threshold = clus_th

# Audio Loader

In [5]:
def load_audio_for_pyannote(path):
    
    #.wav to Tensor 
    
    waveform, sr = torchaudio.load(path)
    
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        waveform = resampler(waveform)
        
    return waveform, 16000

# Output Speaker Labelling
takes the annotations of hte pipeline output, iterates through the speakers according to the order they appeared relative to time and then renames the speaker labels in order of appearance + in the desired submission format

returns a list of dicts which can easily be json.dump()'d  

update: keep min_duration_off to 0.1 so that the VAD seg model picks up on small pauses, stitch pauses lesser than 1.5s together

In [6]:
def get_chronological_segments(output_obj):
    ann = output_obj.speaker_diarization
    mapping = {}
    next_id = 1
    raw_turns = []

    for turn, _, speaker in ann.itertracks(yield_label=True):
        if speaker not in mapping:
            mapping[speaker] = f"SPEAKER_{next_id}"
            next_id += 1
        
        raw_turns.append({
            "start": float(turn.start),
            "end": float(turn.end),
            "id": mapping[speaker]
        })

    if not raw_turns:
        return []

    stitched = []
    curr = raw_turns[0].copy()

    for i in range(1, len(raw_turns)):
        nxt = raw_turns[i]
        
        if (nxt["id"] == curr["id"]) and (nxt["start"] - curr["end"] <= min_stitch):
            curr["end"] = nxt["end"]  # Weld the block
        else:
            stitched.append(curr)
            curr = nxt.copy()
            
    stitched.append(curr)

    return [{
        "start_time": round(s["start"], 2),
        "end_time": round(s["end"], 2),
        "speaker_id": s["id"]
    } for s in stitched]

# Final Diarization on the files + CSV

Sanity Check Version. Only for checking 1 file's output, while testing hyperparam tunes

In [7]:
"""test_files = all_files[:1] 
sanity_rows = []


print(f"sanity check for: {test_files[0]}")

filename = test_files[0]
full_file_path = os.path.join(wav_path, filename)
stem = os.path.splitext(filename)[0]   

try:
    waveform, sample_rate = load_audio_for_pyannote(full_file_path)
    
    audio_payload = {
        "waveform": waveform, 
        "sample_rate": sample_rate,
        "uri": stem
    }

    dia = pipeline(audio_payload, min_speakers=1, max_speakers=20)
    pred = get_chronological_segments(dia)
    json_str = json.dumps(pred, ensure_ascii=False)
    
    sanity_rows.append({
        "filename": stem,
        "diarization": json_str,
    })
    torch.cuda.empty_cache()
    gc.collect()

except Exception as e:
    print(f"Error on {stem}: {e}")
    sanity_rows.append({"filename": stem, "diarization": "[]"})


sanity_df = pd.DataFrame(sanity_rows)
sanity_df.to_csv("sanity_check.csv", index=False)
    
print("\n sanity_check.csv created.")
print(f"Segments: {len(pred)} | JSON Chars: {len(json_str)}")
    
if len(json_str) > 32767:
    print("JSON TOO BIG lol ")
else:
    print("JSON fits 7u7 ")

torch.cuda.empty_cache()
gc.collect()"""

'test_files = all_files[:1] \nsanity_rows = []\n\n\nprint(f"sanity check for: {test_files[0]}")\n\nfilename = test_files[0]\nfull_file_path = os.path.join(wav_path, filename)\nstem = os.path.splitext(filename)[0]   \n\ntry:\n    waveform, sample_rate = load_audio_for_pyannote(full_file_path)\n    \n    audio_payload = {\n        "waveform": waveform, \n        "sample_rate": sample_rate,\n        "uri": stem\n    }\n\n    dia = pipeline(audio_payload, min_speakers=1, max_speakers=20)\n    pred = get_chronological_segments(dia)\n    json_str = json.dumps(pred, ensure_ascii=False)\n    \n    sanity_rows.append({\n        "filename": stem,\n        "diarization": json_str,\n    })\n    torch.cuda.empty_cache()\n    gc.collect()\n\nexcept Exception as e:\n    print(f"Error on {stem}: {e}")\n    sanity_rows.append({"filename": stem, "diarization": "[]"})\n\n\nsanity_df = pd.DataFrame(sanity_rows)\nsanity_df.to_csv("sanity_check.csv", index=False)\n    \nprint("\n sanity_check.csv created.")

All Files Processing + Output

In [8]:
pred_rows = []

for filename in tqdm(all_files, desc="Diarizing"):

    full_file_path = os.path.join(wav_path, filename)
    stem = os.path.splitext(filename)[0]   
    
    try:
        waveform, sample_rate = load_audio_for_pyannote(full_file_path)
        
        audio_payload = {
            "waveform": waveform, 
            "sample_rate": sample_rate,
            "uri": stem
        }
    
        dia = pipeline(audio_payload, min_speakers=1, max_speakers=20)
        pred = get_chronological_segments(dia)
        
        pred_rows.append({
            "filename": stem,
            "diarization": json.dumps(pred, ensure_ascii=False),
        })
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"Error on {stem}: {e}")
        pred_rows.append({"filename": stem, "diarization": "[]"})

sub_df = pd.DataFrame(pred_rows)
sub_df.to_csv(OUT_PATH, index=False)

print(f"Done")
sub_df.head()

Diarizing:   0%|          | 0/14 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.a

Done


,filename,diarization
0,test_010,"[{""start_time"": 13.24, ""end_time"": 44.7, ""spea..."
1,test_012,"[{""start_time"": 0.03, ""end_time"": 90.13, ""spea..."
2,test_016,"[{""start_time"": 35.52, ""end_time"": 40.78, ""spe..."
3,test_018,"[{""start_time"": 0.03, ""end_time"": 30.29, ""spea..."
4,test_019,"[{""start_time"": 1.95, ""end_time"": 6.27, ""speak..."
